# Patent phrase matching: fine-tuning a Transformer cross-encoder

This notebook completes the lab in `project_brief.md`. It builds on the course starter notebook `patent_phrase_starter.ipynb` (same model, input format, validation split and training settings), and covers every task:

| Section | Brief task |
| --- | --- |
| 4 | Task 1: reproduce the fine-tuned baseline |
| 5 | Task 2: baseline without fine-tuning (bi-encoder) |
| 6 | Task 3: controlled experiments (context representation, learning rate) + seed rerun for noise |
| 7 | Choose the final model by validation score, write `submission.csv` and `val_predictions.csv` |
| 8 | Task 4: error analysis |
| 9 | Task 5: leaderboard scores |

**Kaggle settings:** GPU T4 or P100, **Internet off**. Attached inputs:
1. Competition data: *Add Input → Competitions → U.S. Patent Phrase to Phrase Matching*.
2. The output of `download_weights.ipynb` run with `MODEL_ID = "microsoft/deberta-v3-small"`.
3. The output of `download_weights.ipynb` run with `MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"` (for Task 2).

The whole notebook (6 fine-tuning runs) takes roughly 30–45 minutes on a T4, well inside the competition's runtime limit, so the submitted version reruns everything and the final model is picked by validation score automatically.

## 1. Setup

In [ ]:
import gc
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModel, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

transformers.logging.set_verbosity_error()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

## 2. Configuration
Baseline settings are exactly those of the starter notebook. `SEED` and `VAL_FRACTION` are never changed, so every run uses the same validation split.

In [ ]:
SEED = 42
VAL_FRACTION = 0.2
MODEL_ID = "microsoft/deberta-v3-small"
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"   # Task 2 bi-encoder
MAX_LEN = 64
EPOCHS = 3
LR = 2e-5
BATCH_SIZE = 32
OUT_DIR = "/kaggle/working"


def find_one(pattern):
    """Return the directory of the first file under /kaggle/input that matches the pattern."""
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    return os.path.dirname(hits[0]) if hits else None


def local_or_hub(model_id):
    """Use the attached offline copy if there is one, otherwise the Hub name (needs internet)."""
    return find_one(f"{model_id.split('/')[-1]}/config.json") or model_id


DATA_DIR = find_one("us-patent-phrase-to-phrase-matching/train.csv") or find_one("train.csv")
assert DATA_DIR, "Competition data not found: add it with Add Input -> Competitions."
print("data:", DATA_DIR)
print("cross-encoder:", local_or_hub(MODEL_ID))
print("bi-encoder:", local_or_hub(EMBED_MODEL_ID))

## 3. Data and the anchor-grouped validation split

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test:", test.shape, "| distinct anchors:", train.anchor.nunique())
print(train.score.value_counts(normalize=True).sort_index().round(3))

splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

### Model input
As in the starter: `[CLS] anchor [SEP] target; context [SEP]`. The `context_mode` argument lets the context experiment change only this part:
- `"section"` (baseline): CPC section name, e.g. *physics*
- `"code"`: the raw CPC code, e.g. *G02*
- `"none"`: no context at all

In [ ]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def add_inputs(df, context_mode="section"):
    df = df.copy()
    df["text_a"] = df["anchor"]
    if context_mode == "section":
        df["text_b"] = df["target"] + "; " + df["context"].str[0].map(CPC_SECTIONS)
    elif context_mode == "code":
        df["text_b"] = df["target"] + "; " + df["context"]
    else:  # "none"
        df["text_b"] = df["target"]
    return df


print(add_inputs(trn.head(2), "section")[["text_a", "text_b"]].values)
print(add_inputs(trn.head(2), "code")[["text_a", "text_b"]].values)

### One training run
`run_experiment` is the starter's tokenization + fine-tuning + prediction code wrapped in a function, so each experiment changes exactly one argument. It returns the validation Pearson, training time, peak GPU memory, and the validation and test predictions.

In [ ]:
def load_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    # from the starter: transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
    tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])
    return tokenizer


def to_dataset(df, tokenizer, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}


def run_experiment(model_id=MODEL_ID, context_mode="section", lr=LR, epochs=EPOCHS,
                   batch_size=BATCH_SIZE, seed=SEED):
    transformers.set_seed(seed)
    model_path = local_or_hub(model_id)
    tokenizer = load_tokenizer(model_path)
    ds_trn = to_dataset(add_inputs(trn, context_mode), tokenizer)
    ds_val = to_dataset(add_inputs(val, context_mode), tokenizer)
    ds_test = to_dataset(add_inputs(test, context_mode), tokenizer, with_labels=False)

    # fp32 weights are needed for fp16 mixed precision (see the starter notebook)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=1, dtype=torch.float32)
    args = TrainingArguments(
        output_dir=f"{OUT_DIR}/trainer_tmp", num_train_epochs=epochs, learning_rate=lr,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=2 * batch_size,
        warmup_steps=0.1, weight_decay=0.01, eval_strategy="epoch", save_strategy="no",  # warmup_steps < 1 is a ratio
        logging_steps=100, fp16=torch.cuda.is_available(), report_to="none", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                      data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start = time.time()
    trainer.train()
    train_min = (time.time() - start) / 60

    val_pred = trainer.predict(ds_val).predictions.reshape(-1)
    test_pred = trainer.predict(ds_test).predictions.reshape(-1)
    result = {
        "val_pearson": np.corrcoef(val_pred, val["score"])[0, 1],
        "train_min": train_min,
        "gpu_mem_gb": torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0,
        "trainable_params_m": sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6,
        "val_pred": val_pred,
        "test_pred": test_pred,
    }
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"val Pearson {result['val_pearson']:.4f} | train time {train_min:.1f} min")
    return result


results = {}   # setting name -> result of run_experiment

## 4. Task 1: Reproduce the fine-tuned baseline
Starter settings unchanged: `deberta-v3-small`, section-name context, LR 2e-5, 3 epochs, batch size 32.

In [ ]:
results["baseline (starter)"] = run_experiment()

**Why the split holds out whole anchors.** Each of the ~733 anchors appears with ~50 different targets. With a random row split almost every validation anchor would also be in training, so the model could score well by memorising how each anchor tends to be rated (its typical score level and which targets are close to it) instead of learning phrase similarity in general. The validation Pearson would then be optimistically high and would not predict performance on the test set, whose anchors are mostly unseen. It would also mislead experiment choices, because settings that memorise better would look better. Grouping by anchor makes validation look like the real task: new anchors, never seen in training.

## 5. Task 2: Baseline without fine-tuning (bi-encoder)
`all-MiniLM-L6-v2` (Reimers & Gurevych, 2019; Apache-2.0 license) is a pretrained sentence-embedding model. Anchor and target are encoded **separately**, mean-pooled into one vector each (the pooling this model was trained with), and the cosine similarity of the two vectors is the prediction. No training on the competition data.

This is plain `transformers` code doing what `sentence_transformers.SentenceTransformer.encode` does, so no extra package is needed offline.

In [ ]:
def embed(texts, tokenizer, encoder, batch_size=256):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(texts[i:i + batch_size], padding=True, truncation=True, max_length=32,
                          return_tensors="pt").to(device)
        with torch.no_grad():
            hidden = encoder(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).float()
        mean = (hidden * mask).sum(1) / mask.sum(1)           # mean pooling over real tokens
        vecs.append(torch.nn.functional.normalize(mean, dim=-1).cpu())
    return torch.cat(vecs)


embed_path = local_or_hub(EMBED_MODEL_ID)
embed_tok = AutoTokenizer.from_pretrained(embed_path)
embed_model = AutoModel.from_pretrained(embed_path).to(device).eval()

start = time.time()
cos_sim = (embed(val["anchor"].tolist(), embed_tok, embed_model) *
           embed(val["target"].tolist(), embed_tok, embed_model)).sum(1).numpy()
bi_pearson = np.corrcoef(cos_sim, val["score"])[0, 1]
print(f"Bi-encoder (no fine-tuning) val Pearson: {bi_pearson:.4f} | {time.time() - start:.1f} s")

del embed_model
gc.collect()
torch.cuda.empty_cache()

pd.DataFrame({
    "model": ["all-MiniLM-L6-v2 bi-encoder, cosine, no training",
              "deberta-v3-small cross-encoder, fine-tuned (Task 1)"],
    "val_pearson": [bi_pearson, results["baseline (starter)"]["val_pearson"]],
}).round(4)

**Bi-encoder vs cross-encoder.** A *bi-encoder* turns each phrase into a vector on its own, then compares the vectors with cosine similarity. The two phrases never see each other inside the network, so the comparison happens only through one fixed-size summary per phrase. It is fast and the vectors can be pre-computed, but it cannot use the CPC context, and "similar" means whatever the model learned in pre-training (general topical relatedness, e.g. on NLI and web question pairs), not the patent raters' scale.

A *cross-encoder* reads `[CLS] anchor [SEP] target; context [SEP]` as one sequence, so self-attention can align tokens of the two phrases directly (e.g. *opc* ↔ *photoconductor*) and condition on the context. A regression head on top is trained with MSE on the 0–1 labels, so the model learns the raters' notion of similarity, including that a hypernym, a part, or an antonym gets a specific score.

**What fine-tuning added.** Compare the two numbers above: the bi-encoder already gets a clearly positive correlation from pre-training alone, which shows general-purpose embeddings capture much of the phrase relatedness. The gap up to the fine-tuned cross-encoder is what training on ~29k labelled pairs adds: learning the label scale, the patent-specific meanings, and using the context. Note the comparison mixes two changes (architecture and fine-tuning), so the gap is the combined effect of both.

## 6. Task 3: Controlled experiments
Each run changes **one** argument from the baseline; everything else (model, split, epochs, batch size, seed) stays the same.

- **Experiment A – context representation:** `none` and raw CPC `code`, vs. section name (baseline).
- **Experiment B – learning rate:** 1e-5 and 5e-5, vs. 2e-5 (baseline). Three values of one setting, as the brief allows.
- **Noise check:** the baseline rerun with a different training seed (43). The split does not change, since it uses its own `random_state=SEED`. The gap between the two baseline runs tells us how big a difference has to be before it means anything.

In [ ]:
results["baseline, seed 43 (noise check)"] = run_experiment(seed=43)
results["context: none"] = run_experiment(context_mode="none")
results["context: raw CPC code"] = run_experiment(context_mode="code")
results["LR 1e-5"] = run_experiment(lr=1e-5)
results["LR 5e-5"] = run_experiment(lr=5e-5)
# Optional checkpoint experiment (attach its download_weights output first, it is slower and uses more memory):
# results["checkpoint: deberta-v3-base"] = run_experiment(model_id="microsoft/deberta-v3-base")

### Results table
`noise` is the larger of 0.005 (the brief's rule of thumb) and the gap between the two baseline seeds. A difference smaller than that is reported as "within noise".

In [ ]:
base = results["baseline (starter)"]["val_pearson"]
noise = max(0.005, abs(base - results["baseline, seed 43 (noise check)"]["val_pearson"]))


def interpret(name, r):
    if name == "baseline (starter)":
        return "reference run"
    if "seed" in name:
        return f"same settings, other seed: run-to-run noise ~{abs(r - base):.4f}"
    diff = r - base
    if abs(diff) <= noise:
        return f"{diff:+.4f} vs baseline: within noise, no real difference"
    return f"{diff:+.4f} vs baseline: {'better' if diff > 0 else 'worse'} than baseline (beyond noise)"


table = pd.DataFrame([{
    "setting changed": name,
    "val_pearson": round(r["val_pearson"], 4),
    "train_min": round(r["train_min"], 1),
    "gpu_mem_gb": round(r["gpu_mem_gb"], 2),
    "interpretation": interpret(name, r["val_pearson"]),
} for name, r in results.items()])
print(f"noise threshold = {noise:.4f}")
table

**What the results show** (Kaggle T4 run, version 3):
- *Noise:* the two baseline seeds gave 0.8170 and 0.8173, so seed noise here was only ~0.0003, and the brief's 0.005 is the threshold that applies.
- *Context:* no context (0.8144, −0.0026) and the raw CPC code (0.8139, −0.0031) are both within noise of the section name. On this split the context representation made **no measurable difference**. This is not because context is constant: validation anchors appear with 2.6 CPC classes on average, and 54% span more than one section. A likely reason is that for most pairs the phrases themselves already signal the field, so the context rarely changes the rating. We can't conclude that context is useless in general.
- *Learning rate:* 1e-5 (0.8000, −0.017) is clearly worse. With 3 epochs it under-trains, and its epoch-1 score (0.772) was the lowest of all runs. 5e-5 (0.8212, +0.0042) was the best single run, but the gain is **within the 0.005 noise threshold**, so it is not shown to be better than 2e-5 without more seeds.
- Training time was ~5.2–5.6 min per run (3 epochs) for every setting, because none of these changes affects model size.

## 7. Final model and submission
The final model is the run with the **highest validation Pearson** (never chosen by leaderboard score). Its test predictions were already computed inside `run_experiment`, so there is no retraining. The seed-43 rerun is excluded because it is a noise check, not a different setting.

In [ ]:
candidates = {k: v for k, v in results.items() if "seed" not in k}
best_name = max(candidates, key=lambda k: candidates[k]["val_pearson"])
best = results[best_name]
print(f"Final model: {best_name} | val Pearson {best['val_pearson']:.4f}")

val["pred"] = best["val_pred"]
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

submission = pd.DataFrame({"id": test["id"], "score": np.clip(best["test_pred"], 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

## 8. Task 4: Error analysis (best model, `val_predictions.csv`)

### 8.1 Pearson per CPC section

In [ ]:
vp = pd.read_csv(f"{OUT_DIR}/val_predictions.csv")
vp["section"] = vp["context"].str[0]
per_section = vp.groupby("section").apply(
    lambda g: pd.Series({"rows": len(g), "pearson": np.corrcoef(g["pred"], g["score"])[0, 1],
                         "mean_score": g["score"].mean()}), include_groups=False)
per_section["name"] = per_section.index.map(CPC_SECTIONS)
per_section.sort_values("pearson").round(3)

**Observed:** the hardest sections are **G physics (0.701)**, C chemistry (0.766) and A human necessities (0.783). The easiest are B performing operations/transporting (0.900) and F mechanical engineering (0.856). G covers optics, computing and measurement, where everyday words take technical meanings (*imaging axis*, *identity*, *position*), and several of the largest errors below are G pairs. C and A contain compound names and biology/medical terms that need domain knowledge. B and F phrases are concrete mechanical parts, where word overlap is a more reliable signal. D (12 anchors) and E (14 anchors) have few validation anchors, so their Pearson values are noisy.

### 8.2 Mean prediction for each true score

In [ ]:
per_score = vp.groupby("score")["pred"].agg(["count", "mean", "std"]).round(3)
print(per_score)

plt.figure(figsize=(5, 4))
plt.errorbar(per_score.index, per_score["mean"], yerr=per_score["std"], fmt="o-", capsize=4, label="mean prediction ± 1 std")
plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect calibration")
plt.xlabel("true score")
plt.ylabel("predicted score")
plt.title("Figure 1: mean prediction per true score (validation)")
plt.legend()
plt.show()

**Observed:** mean predictions are 0.099 / 0.302 / 0.493 / 0.656 / 1.012 for true scores 0 / 0.25 / 0.5 / 0.75 / 1. The middle scores are well calibrated. There is **some compression**: true 0 is predicted slightly high (0.10) and true 0.75 clearly low (0.66, the biggest gap), so 0.75 pairs overlap with 0.5 pairs. Score 1.0 is *not* compressed, because those pairs are mostly exact or near-exact matches that are easy to recognise. This fits the label imbalance: 0.25 and 0.5 make up 65% of the labels and 0.75 only 11%, so under MSE the model is pulled towards the common middle levels.

### 8.3 The 10 largest errors

In [ ]:
vp["abs_error"] = (vp["pred"] - vp["score"]).abs()
vp["shared_words"] = [len(set(a.split()) & set(t.split())) for a, t in zip(vp["anchor"], vp["target"])]
top10 = vp.sort_values("abs_error", ascending=False).head(10)
pd.set_option("display.max_colwidth", 60)
top10[["anchor", "target", "context", "score", "pred", "abs_error", "shared_words"]].round(3)

**Failure types observed in the 10 largest errors:**
1. **Same words, different meaning (over-prediction, 7 of 10).** *punch face* → *face punch* (true 0, pred 0.90), *imaging axis* → *imaging of axis* / *image of axis* (0 vs 0.84 / 0.77), *boom hydraulic cylinder* → *boom a hydraulic cylinder* (0 vs 0.82), *secure to guard* → *security guard*, *return structure* → *return to structure*, *activating position* → *active region*. The model leans on word overlap. It misses that word order or small function words change the meaning (a *punch face* is the face of a punch tool; a *face punch* is a blow to the face).
2. **Related without shared words (under-prediction, 3 of 10).** *end pins* → *wood pin* (true 0.75, pred −0.01), *breech mechanism* → *breech loading naval gun* (0.75 vs 0.03), *whose identity* → *customer id* (0.75 vs 0.06). Seeing that these are related needs domain knowledge or an abbreviation (*id* = identity) that the model did not learn.
3. **Debatable labels.** Some "true" labels are questionable. *boom hydraulic cylinder* vs *boom a hydraulic cylinder* look nearly identical, and *imaging of axis* scoring 0 against *imaging axis* is hard to defend. *end pins* → *wood pin* at 0.75 is also surprising. These look like rater judgements on near-duplicate strings, so part of this "error" is label noise rather than model failure.

## 9. Task 5: Late submission and leaderboard scores
After *Save Version → Save & Run All* finishes, submit this version from the competition's **Late Submission** button. Then record every submission:

| Submission | Final model (chosen by validation) | Validation Pearson | Public score | Private score |
| --- | --- | --- | --- | --- |
| 1 (notebook version 3, ref 57807770) | LR 5e-5 | 0.8212 | 0.7961 | 0.8157 |
| 2 (notebook version 4, ref 57808203, same code resubmitted) | LR 5e-5 | 0.8212 | 0.7961 | 0.8157 |

**Comparison.** Validation 0.8212, private 0.8157 and public 0.7961 are close. Private is only 0.0055 below validation, so the anchor-grouped split gave a realistic estimate. Public is 0.02 lower, which is most likely sampling noise: it is scored on a smaller part of the hidden test set. The two submissions came from identical code and got identical scores.

**Why validation, public and private can differ.**
- *Different anchors and sections:* the test set has its own anchors and its own mix of CPC sections; Section 8.1 shows performance varies by section, so a different mix shifts the score.
- *Test-set size:* the public part is a small slice of the hidden test set, so its Pearson has more sampling noise than the private score.
- *Selection on validation:* the final model was picked as the best of several runs on the same validation set, so its validation score is slightly optimistic (winner's curse); some of its advantage may be noise.
- *Training data:* the final model is trained on 80% of `train.csv` (the other 20% is the validation split).